In [8]:
# Mount Drive and load out_features.jsonl

from google.colab import drive
drive.mount('/content/drive')

!pip -q install xgboost scipy scikit-learn pandas numpy tqdm

import os, json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# EDIT THIS PATH if your file lives somewhere else
JSONL_PATH = '/content/drive/MyDrive/Colab Notebooks/out_features.jsonl'

assert os.path.exists(JSONL_PATH), f'File not found: {JSONL_PATH}'
print(f'Found: {JSONL_PATH}  ({os.path.getsize(JSONL_PATH)/1e6:.1f} MB)')

# Stream the JSONL once and rebuild records_df + fire_lists
records    = []
fire_lists = []

with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc='loading'):
        line = line.strip()
        if not line: continue
        try:
            r = json.loads(line)
        except Exception:
            continue

        label = (r.get('label') or '').lower()
        meas  = r.get('measurements') or {}
        feats = r.get('features') or []

        seen_ids = sorted({fo.get('id') for fo in feats if fo.get('id')})

        rec = {
            'id': r.get('id'),
            'label': label,
            'source': r.get('source'),
            'n_features_fired': len(seen_ids),
        }
        for k, v in meas.items():
            if isinstance(v, bool):
                rec[f'm_{k}'] = int(v)
            elif isinstance(v, (int, float)):
                rec[f'm_{k}'] = v
            elif isinstance(v, list):
                rec[f'm_{k}'] = len(v)
            elif isinstance(v, dict):
                rec[f'm_{k}'] = len(v)
        records.append(rec)
        fire_lists.append(seen_ids)

records_df = pd.DataFrame(records)
print(f'\nLoaded {len(records_df):,} records')
print(records_df['label'].value_counts())
print(f'Numeric measurement columns: {sum(c.startswith("m_") for c in records_df.columns)}')
print(f'Distinct fired-feature IDs:  {len({fid for lst in fire_lists for fid in lst})}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: /content/drive/MyDrive/Colab Notebooks/out_features.jsonl  (1133.4 MB)


loading: 0it [00:00, ?it/s]


Loaded 80,000 records
label
phishing    40000
benign      40000
Name: count, dtype: int64
Numeric measurement columns: 21
Distinct fired-feature IDs:  36
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: /content/drive/MyDrive/Colab Notebooks/out_features.jsonl  (1133.4 MB)


loading: 0it [00:00, ?it/s]


Loaded 80,000 records
label
phishing    40000
benign      40000
Name: count, dtype: int64
Numeric measurement columns: 21
Distinct fired-feature IDs:  36


In [9]:
!pip -q install xgboost

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Restrict to labelled rows in case there's any 'other' bucket
mask = records_df['label'].isin(['phishing','benign']).values
y    = (records_df.loc[mask, 'label']=='phishing').astype(int).values

# 1) Binary fired-feature matrix (one column per feature_id)
all_fids = sorted({fid for lst in fire_lists for fid in lst})
fid_to_idx = {fid: i for i, fid in enumerate(all_fids)}

rows_, cols_ = [], []
for ridx, lst in enumerate([fl for fl, m in zip(fire_lists, mask) if m]):
    for fid in lst:
        rows_.append(ridx); cols_.append(fid_to_idx[fid])
X_fire = sp.csr_matrix((np.ones(len(rows_), dtype=np.int8), (rows_, cols_)),
                       shape=(mask.sum(), len(all_fids)))

# 2) Numeric measurements matrix
num_cols = [c for c in records_df.columns if c.startswith('m_')]
X_num = records_df.loc[mask, num_cols].astype(float)
X_num = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_num),
                     columns=num_cols, index=X_num.index)

# 3) Combine
X_full = sp.hstack([X_fire, sp.csr_matrix(X_num.values)]).tocsr()
all_columns = all_fids + num_cols

print('Full feature matrix:', X_full.shape)
print(f'  - {len(all_fids)} binary fired features')
print(f'  - {len(num_cols)} numeric measurements')
print(f'  - {len(y)} labelled rows')

Full feature matrix: (80000, 57)
  - 36 binary fired features
  - 21 numeric measurements
  - 80000 labelled rows
Full feature matrix: (80000, 57)
  - 36 binary fired features
  - 21 numeric measurements
  - 80000 labelled rows


In [10]:
# Features I flagged as likely capture/source artifacts in the audit
suspect_features = [
    # Binary fired features that fire much more on benign (likely Tranco-only artifacts)
    'navigation.functional_internal_links',
    'iframe.hidden_iframe',
    'support.contact_domain_match',
    'link.brand_text_domain_mismatch',
]

# Numeric measurements that look like "page completeness" rather than phishing signal
suspect_measurements = [
    'm_visible_text_length',
    'm_internal_nav_candidate_count',
    'm_anchor_count',
    'm_script_src_sample_count',
    'm_image_src_sample_count',
    'm_stylesheet_href_sample_count',
    'm_favicon_count',
    'm_contact_anchor_domains',
    'm_external_anchor_count',
]

suspect_all = suspect_features + suspect_measurements
suspect_idx = [i for i, c in enumerate(all_columns) if c in suspect_all]
keep_idx    = [i for i, c in enumerate(all_columns) if c not in suspect_all]

print(f'Suspect features to remove: {len(suspect_idx)}')
for c in suspect_all:
    present = '✓' if c in all_columns else '✗ NOT FOUND'
    print(f'  {present}  {c}')
print(f'\nKept columns: {len(keep_idx)} / {len(all_columns)}')

Suspect features to remove: 13
  ✓  navigation.functional_internal_links
  ✓  iframe.hidden_iframe
  ✓  support.contact_domain_match
  ✓  link.brand_text_domain_mismatch
  ✓  m_visible_text_length
  ✓  m_internal_nav_candidate_count
  ✓  m_anchor_count
  ✓  m_script_src_sample_count
  ✓  m_image_src_sample_count
  ✓  m_stylesheet_href_sample_count
  ✓  m_favicon_count
  ✓  m_contact_anchor_domains
  ✓  m_external_anchor_count

Kept columns: 44 / 57
Suspect features to remove: 13
  ✓  navigation.functional_internal_links
  ✓  iframe.hidden_iframe
  ✓  support.contact_domain_match
  ✓  link.brand_text_domain_mismatch
  ✓  m_visible_text_length
  ✓  m_internal_nav_candidate_count
  ✓  m_anchor_count
  ✓  m_script_src_sample_count
  ✓  m_image_src_sample_count
  ✓  m_stylesheet_href_sample_count
  ✓  m_favicon_count
  ✓  m_contact_anchor_domains
  ✓  m_external_anchor_count

Kept columns: 44 / 57


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_full, y, test_size=0.30,
                                            stratify=y, random_state=42)
X_va, X_te, y_va, y_te   = train_test_split(X_tmp, y_tmp, test_size=0.50,
                                            stratify=y_tmp, random_state=42)
print(f'train={X_tr.shape[0]:,}  val={X_va.shape[0]:,}  test={X_te.shape[0]:,}')

def train_eval(X_tr, y_tr, X_va, y_va, X_te, y_te, label):
    clf = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.1,
        tree_method='hist', n_jobs=-1, random_state=0, eval_metric='logloss')
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    pred = clf.predict(X_te)
    f1   = f1_score(y_te, pred)
    acc  = accuracy_score(y_te, pred)
    print(f'\n=== {label} ===')
    print(f'  test F1:       {f1:.4f}')
    print(f'  test accuracy: {acc:.4f}')
    print(classification_report(y_te, pred, target_names=['benign','phishing']))
    print('confusion matrix [[TN FP] [FN TP]]:')
    print(confusion_matrix(y_te, pred))
    return clf, f1, acc, pred

train=56,000  val=12,000  test=12,000
train=56,000  val=12,000  test=12,000


In [12]:
# Run 1: ALL features
clf_full, f1_full, acc_full, _ = train_eval(
    X_tr, y_tr, X_va, y_va, X_te, y_te,
    'Run 1 — ALL features')

# Run 2: features WITHOUT the suspects
X_tr_k = X_tr[:, keep_idx]
X_va_k = X_va[:, keep_idx]
X_te_k = X_te[:, keep_idx]
clf_clean, f1_clean, acc_clean, _ = train_eval(
    X_tr_k, y_tr, X_va_k, y_va, X_te_k, y_te,
    'Run 2 — suspect features REMOVED')

# Summary
print('\n' + '='*55)
print('ABLATION RESULT')
print('='*55)
print(f'  F1 with all features:        {f1_full:.4f}')
print(f'  F1 with suspects removed:    {f1_clean:.4f}')
print(f'  Drop in F1:                  {f1_full - f1_clean:+.4f}  '
      f'({(f1_full - f1_clean)*100:+.2f} pp)')
print()
print(f'  Accuracy with all features:  {acc_full:.4f}')
print(f'  Accuracy without suspects:   {acc_clean:.4f}')


=== Run 1 — ALL features ===
  test F1:       0.9722
  test accuracy: 0.9723
              precision    recall  f1-score   support

      benign       0.97      0.98      0.97      6000
    phishing       0.98      0.97      0.97      6000

    accuracy                           0.97     12000
   macro avg       0.97      0.97      0.97     12000
weighted avg       0.97      0.97      0.97     12000

confusion matrix [[TN FP] [FN TP]]:
[[5856  144]
 [ 188 5812]]

=== Run 2 — suspect features REMOVED ===
  test F1:       0.9281
  test accuracy: 0.9283
              precision    recall  f1-score   support

      benign       0.93      0.93      0.93      6000
    phishing       0.93      0.92      0.93      6000

    accuracy                           0.93     12000
   macro avg       0.93      0.93      0.93     12000
weighted avg       0.93      0.93      0.93     12000

confusion matrix [[TN FP] [FN TP]]:
[[5591  409]
 [ 451 5549]]

ABLATION RESULT
  F1 with all features:        0.97

In [13]:
# Train a classifier using ONLY the suspect features, predicting SOURCE (not label)
# If accuracy is near 100%, those features mostly encode "which crawler captured this"

src_lab = (records_df.loc[mask, 'source'] == 'phishing_db.website_content').astype(int).values
X_suspect_only = X_full[:, suspect_idx]

X_tr_s, X_tmp_s, y_tr_s, y_tmp_s = train_test_split(
    X_suspect_only, src_lab, test_size=0.30, stratify=src_lab, random_state=42)
X_va_s, X_te_s, y_va_s, y_te_s = train_test_split(
    X_tmp_s, y_tmp_s, test_size=0.50, stratify=y_tmp_s, random_state=42)

clf_cheat = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    tree_method='hist', n_jobs=-1, random_state=0, eval_metric='logloss')
clf_cheat.fit(X_tr_s, y_tr_s, eval_set=[(X_va_s, y_va_s)], verbose=False)
pred_cheat = clf_cheat.predict(X_te_s)
f1_cheat   = f1_score(y_te_s, pred_cheat)
acc_cheat  = accuracy_score(y_te_s, pred_cheat)

print('='*55)
print('CHEATING DETECTOR — predict SOURCE from suspect features')
print('='*55)
print(f'  test accuracy: {acc_cheat:.4f}')
print(f'  test F1:       {f1_cheat:.4f}')
print()
print('Interpretation:')
if acc_cheat > 0.95:
    print('  >0.95 → suspect features almost perfectly identify the source.')
    print('  In this dataset, source = label, so they will appear to be')
    print('  excellent phishing predictors — but they\'re really just')
    print('  predicting which crawler captured the page.')
elif acc_cheat > 0.80:
    print('  0.80-0.95 → suspect features are strong source markers but')
    print('  not perfect. Some real signal mixed with artifact.')
else:
    print('  <0.80 → suspect features don\'t cleanly identify the source.')
    print('  The bias concern was overstated; these features may be okay.')

CHEATING DETECTOR — predict SOURCE from suspect features
  test accuracy: 0.9545
  test F1:       0.9547

Interpretation:
  >0.95 → suspect features almost perfectly identify the source.
  In this dataset, source = label, so they will appear to be
  excellent phishing predictors — but they're really just
  predicting which crawler captured the page.
CHEATING DETECTOR — predict SOURCE from suspect features
  test accuracy: 0.9545
  test F1:       0.9547

Interpretation:
  >0.95 → suspect features almost perfectly identify the source.
  In this dataset, source = label, so they will appear to be
  excellent phishing predictors — but they're really just
  predicting which crawler captured the page.


In [14]:
imp_full = pd.DataFrame({
    'feature': all_columns,
    'gain':    clf_full.feature_importances_
}).sort_values('gain', ascending=False).reset_index(drop=True)

imp_full['is_suspect'] = imp_full['feature'].isin(suspect_all)

print('Top 20 features by gain in the full model:')
print(imp_full.head(20).to_string(index=False))

print('\nHow much of the total gain came from suspect features?')
suspect_gain = imp_full.loc[imp_full['is_suspect'], 'gain'].sum()
total_gain   = imp_full['gain'].sum()
print(f'  suspect-feature gain share: {suspect_gain/total_gain:.1%}')

Top 20 features by gain in the full model:
                           feature     gain  is_suspect
    m_internal_nav_candidate_count 0.219002        True
       brand.title_domain_mismatch 0.061960       False
                  m_redirect_count 0.056796       False
content.coercive_urgency_near_form 0.035474       False
    m_stylesheet_href_sample_count 0.031162        True
         page.low_semantic_content 0.030465       False
              iframe.hidden_iframe 0.026984        True
         link.null_or_void_anchors 0.026164       False
                    m_anchor_count 0.026155        True
            m_password_input_count 0.025999       False
                   m_favicon_count 0.025676        True
                url.http_not_https 0.025147       False
               m_null_anchor_count 0.021881       False
      support.contact_domain_match 0.020502        True
             redirect.cross_domain 0.018463       False
  contact.identity_domain_mismatch 0.016098       False
     